### Kütüphanler

In [ ]:
import os
import yaml
import json
import logging
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
from datetime import datetime

# Stable Baselines3 ve Contrib
from stable_baselines3 import PPO
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor, VecFrameStack, DummyVecEnv
from stable_baselines3.common.utils import set_random_seed

# Grafik Ayarları
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.style.use('seaborn-v0_8-whitegrid')

# Logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("Thesis_Phase3")

# Donanım Seçimi (Mac MPS Desteği Dahil)
def get_optimal_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_optimal_device()
print(f"🚀 Donanım Hızlandırıcı: {DEVICE}")

🚀 Donanım Hızlandırıcı: mps

### Wrappers

In [ ]:
# --- 1. Mac/MPS Uyumluluk Wrapper (Zorunlu) ---
class Float32ObservationWrapper(gym.ObservationWrapper):
    """Mac MPS'in float64 desteklememesi sorununu çözer. Tüm inputları float32 yapar."""
    def __init__(self, env):
        super().__init__(env)
        self.observation_space = gym.spaces.Box(
            low=env.observation_space.low,
            high=env.observation_space.high,
            shape=env.observation_space.shape,
            dtype=np.float32
        )
    def observation(self, observation):
        return np.array(observation, dtype=np.float32)

# --- 2. Fiziksel Manipülasyon (Fault Injection Ready) ---
class RandomDampingWrapper(gym.Wrapper):
    """
    Sürtünmeyi (Damping) değiştirir. 
    'set_damping' metodu sayesinde epizot ortasında (Reset atmadan) müdahale edilebilir.
    """
    def __init__(self, env, min_damping, max_damping):
        super().__init__(env)
        self.min_d = min_damping
        self.max_d = max_damping

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # Her epizot başında rastgele bir fizik katsayısı belirle (Domain Randomization)
        new_damping = self.np_random.uniform(self.min_d, self.max_d)
        self.set_damping(new_damping)
        return self.env.reset(seed=seed, options=options)

    def set_damping(self, value):
        """Dışarıdan veya anlık değişim (Sudden Fault) için çağrılır."""
        if hasattr(self.env.unwrapped, 'model'):
            dof_count = len(self.env.unwrapped.model.dof_damping)
            if isinstance(value, (float, int)):
                val_array = np.full(dof_count, value)
            else:
                val_array = value
            self.env.unwrapped.model.dof_damping[:] = val_array

# --- 3. Context Awareness (Previous Action) ---
class PreviousActionWrapper(gym.Wrapper):
    """
    Gözlem vektörüne bir önceki aksiyonu ekler.
    Input: [Obs_t] -> [Obs_t, Action_{t-1}]
    Bu sayede LSTM: "Ben X torku uyguladım ama Y kadar gittim, demek ki sürtünme Z" çıkarımını yapabilir.
    """
    def __init__(self, env):
        super().__init__(env)
        obs_space = env.observation_space
        act_space = env.action_space
        
        # Yeni observation space tanımı (Mevcut + Aksiyon Boyutu)
        low = np.concatenate([obs_space.low, act_space.low])
        high = np.concatenate([obs_space.high, act_space.high])
        
        self.observation_space = gym.spaces.Box(low=low, high=high, dtype=np.float32)
        self.prev_action = np.zeros(act_space.shape, dtype=np.float32)

    def reset(self, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options)
        self.prev_action = np.zeros(self.env.action_space.shape, dtype=np.float32)
        new_obs = np.concatenate([obs, self.prev_action]).astype(np.float32)
        return new_obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        new_obs = np.concatenate([obs, self.prev_action]).astype(np.float32)
        self.prev_action = action.astype(np.float32) # Bir sonraki adım için sakla
        return new_obs, reward, terminated, truncated, info

### Trainer

In [ ]:
# Notebook Hücresi: Scientific Trainer (Clean & Stable)
import os
import yaml
import json
import logging
import torch
import gymnasium as gym
import numpy as np
from datetime import datetime

# SB3
from stable_baselines3 import PPO
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor, VecFrameStack
from stable_baselines3.common.utils import set_random_seed

# Local
from src.utils.device import get_optimal_device
# Wrapperlarınızın importları (yukarıdaki hücrede tanımlı olduklarını varsayıyorum)
# from src.envs.wrappers... import ...

class ScientificTrainer:
    def __init__(self, config_path: str):
        """
        Config dosyasını diskten okur, log klasörünü hazırlar.
        Tweak mekanizması iptal edildi.
        """
        self.logger = logging.getLogger("ScientificTrainer")
        self.logger.setLevel(logging.INFO)
        
        # 1. Config Dosyasını Oku (Single Source of Truth)
        if not os.path.exists(config_path):
            raise FileNotFoundError(f"❌ Config dosyası bulunamadı: {config_path}")
            
        with open(config_path) as f:
            self.config = yaml.safe_load(f)
            
        # 2. Run ID ve Klasör Yapısı
        self.timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.run_id = f"{self.config['experiment']['name']}_{self.timestamp}"
        self.device = get_optimal_device()
        
        # Log Klasörü (Notebook konumuna göre ../data/logs)
        self.log_dir = os.path.join("..", "data", "logs", self.run_id)
        os.makedirs(self.log_dir, exist_ok=True)
        
        # 3. Config'in Kopyasını Arşivle (Evidence)
        # Orijinal dosyayı değil, okuduğumuz içeriği JSON olarak dump ediyoruz.
        with open(os.path.join(self.log_dir, "config.json"), "w") as f:
            json.dump(self.config, f, indent=4)
            
        self.print_obsidian_header(config_path)

    def print_obsidian_header(self, original_path):
        print("\n" + "="*40)
        print("📋 OBSIDIAN LAB NOTE HEADER")
        print("="*40)
        print(f"run_id: \"{self.run_id}\"")
        print(f"source_config: \"{original_path}\"")
        print(f"device: \"{self.device}\"")
        print("="*40 + "\n")

    def _make_env_factory(self, seed, rank, wrapper_conf, force_cpu=False):
        def _init():
            env = gym.make(self.config['env']['id'])
            
            # Device Handling
            device_type = "cpu" if force_cpu else str(self.device)
            if device_type == "mps": 
                env = Float32ObservationWrapper(env)
            
            # Physics
            env = RandomDampingWrapper(
                env, 
                min_damping=wrapper_conf['min_damping'], 
                max_damping=wrapper_conf['max_damping']
            )
            # Context
            env = PreviousActionWrapper(env)
            
            env.reset(seed=seed + rank)
            return env
        return _init

    def train_variant(self, variant_name):
        print(f"\n{'='*60}")
        print(f"🚀 EĞİTİM BAŞLIYOR: {variant_name}")
        print(f"{'='*60}")
        
        seeds = self.config['training']['seeds']
        wrapper_conf = self.config['env']['wrappers'][0]['args']
        train_timesteps = self.config['training']['total_timesteps']
        
        trained_model_paths = []

        # Model Seçimi ve Parametre Temizliği
        if variant_name == "LSTM":
            ModelClass = RecurrentPPO
            policy_type = "MlpLstmPolicy"
            current_device = self.device
            force_cpu = False
            policy_kwargs = self.config['hyperparameters']['policy_kwargs']
            print(f"   🧠 Model: RecurrentPPO (GPU/MPS)")
        else:
            ModelClass = PPO
            policy_type = "MlpPolicy"
            current_device = "cpu" # MLP için CPU daha hızlı/stabil
            force_cpu = True
            # LSTM parametrelerini temizle
            base_kwargs = self.config['hyperparameters']['policy_kwargs'].copy()
            exclude = ['lstm_hidden_size', 'n_lstm_layers', 'shared_lstm', 'enable_critic_lstm']
            policy_kwargs = {k: v for k, v in base_kwargs.items() if k not in exclude}
            print(f"   🧠 Model: Standard PPO (CPU Optimized)")

        for seed in seeds:
            set_random_seed(seed)
            
            # Ortam Kurulumu
            env = SubprocVecEnv([
                self._make_env_factory(seed, i, wrapper_conf, force_cpu) 
                for i in range(self.config['env']['n_envs'])
            ])
            
            if variant_name == "FrameStack":
                env = VecFrameStack(env, n_stack=4)
                print(f"   🛠️ FrameStack: Aktif (4 frames)")
            
            env = VecMonitor(env, filename=os.path.join(self.log_dir, f"{variant_name}_seed_{seed}_monitor.csv"))
            
            model = ModelClass(
                policy=policy_type,
                env=env,
                verbose=1, # Tablo çıktısı için 1 olmalı
                device=current_device,
                tensorboard_log=self.log_dir,
                learning_rate=self.config['hyperparameters']['learning_rate'],
                n_steps=self.config['hyperparameters']['n_steps'],
                batch_size=self.config['hyperparameters']['batch_size'],
                gamma=self.config['hyperparameters']['gamma'],
                gae_lambda=self.config['hyperparameters']['gae_lambda'],
                ent_coef=self.config['hyperparameters']['ent_coef'],
                policy_kwargs=policy_kwargs
            )

            print(f"   🌱 Seed {seed} eğitiliyor... ({train_timesteps} steps)")
            
            # DÜZELTME: progress_bar=False yaptık. 
            # Ekran titremesini önlemek için sadece log tablosu basacak.
            model.learn(
                total_timesteps=train_timesteps, 
                tb_log_name=f"{variant_name}_seed_{seed}",
                progress_bar=False, 
                log_interval=10 # Her 10 update'de (20k adım) bir tablo basar, spam yapmaz.
            )
            
            save_path = os.path.join(self.log_dir, f"final_model_{variant_name}_seed_{seed}")
            model.save(save_path)
            trained_model_paths.append(save_path)
            print(f"   ✅ Model Kaydedildi: {save_path}")
            
            env.close()
            
        return trained_model_paths

### Config Oluşturma ve Training

In [4]:
# 1. Config Dosyasını Diske Yaz (Tekrarlanabilirlik İçin)
config_dir = "../configs/phase_3_1"
os.makedirs(config_dir, exist_ok=True)
config_path = os.path.join(config_dir, "reacher_experiment_final.yaml")

yaml_content = """
experiment:
  name: "Phase-3.1-Comparison"
  description: "Comparing Vanilla, FrameStack, and LSTM on Reacher-v5"

env:
  id: "Reacher-v5"
  n_envs: 2
  wrappers:
    - name: "RandomDampingWrapper"
      args: {min_damping: 0.5, max_damping: 2.0}

training:
  algo: "Mixed"
  total_timesteps: 500000  # Tez için bunu 500k veya 1M yapın
  seeds: [42] # Hızlı demo için tek seed, gerçekte [42, 43, 44] yapın

hyperparameters:
  policy_type: "MlpLstmPolicy" # veya MlpPolicy (Otomatik seçiliyor)
  learning_rate: 0.0003
  n_steps: 2048
  batch_size: 64
  gamma: 0.99
  gae_lambda: 0.95
  ent_coef: 0.01
  policy_kwargs:
    # Ortak MLP Ayarları
    net_arch: 
      pi: [64, 64]
      vf: [64, 64]
    # LSTM Ayarları (Sadece RecurrentPPO kullanır)
    lstm_hidden_size: 256
    n_lstm_layers: 1
    shared_lstm: False
    enable_critic_lstm: True
"""

with open(config_path, "w") as f:
    f.write(yaml_content)

# 2. Trainer'ı Başlat
trainer = ScientificTrainer(config_path)


📂 Log: ../data/logs/Phase-3.1-Comparison_20251125_121126


In [5]:

# 3. ÜÇ VARYASYONU DA EĞİT
# Çıktı olarak kaydedilen modellerin yollarını alıyoruz
paths_vanilla = trainer.train_variant("Vanilla")
print("\n🎉 PPO (Vanilla) Eğitimi BAŞARIYLA TAMAMLANDI.")


🚀 MODEL EĞİTİMİ BAŞLIYOR: Vanilla
   ⚡ Donanım: CPU (MLP model için optimize - Uyarı engellendi)


/Users/berkaygurkan/pytorch_m1_env/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/berkaygurkan/pytorch_m1_env/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/berkaygurkan/pytorch_m1_env/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/Users/berkaygurkan/pytorch_m1_env/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


Using cpu device
   🌱 Seed 42 eğitiliyor...
Logging to ../data/logs/Phase-3.1-Comparison_20251125_121126/Vanilla_seed_42_1


Output()

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | -54.5       |
| time/                   |             |
|    fps                  | 2523        |
|    iterations           | 10          |
|    time_elapsed         | 16          |
|    total_timesteps      | 40960       |
| train/                  |             |
|    approx_kl            | 0.015395533 |
|    clip_fraction        | 0.142       |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.05       |
|    explained_variance   | 0.936       |
|    learning_rate        | 0.0003      |
|    loss                 | 4.2         |
|    n_updates            | 90          |
|    policy_gradient_loss | -0.0202     |
|    std                  | 0.659       |
|    value_loss           | 6.73        |
-----------------------------------------


...

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | -10.9       |
| time/                   |             |
|    fps                  | 2390        |
|    iterations           | 40          |
|    time_elapsed         | 68          |
|    total_timesteps      | 163840      |
| train/                  |             |
|    approx_kl            | 0.014890374 |
|    clip_fraction        | 0.14        |
|    clip_range           | 0.2         |
|    entropy_loss         | 0.985       |
|    explained_variance   | 0.934       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.116       |
|    n_updates            | 390         |
|    policy_gradient_loss | -0.00989    |
|    std                  | 0.146       |
|    value_loss           | 0.249       |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | -9.44       |
| time/                   |             |
|    fps                  | 2410        |
|    iterations           | 50          |
|    time_elapsed         | 84          |
|    total_timesteps      | 204800      |
| train/                  |             |
|    approx_kl            | 0.017030077 |
|    clip_fraction        | 0.168       |
|    clip_range           | 0.2         |
|    entropy_loss         | 1.63        |
|    explained_variance   | 0.994       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.0475      |
|    n_updates            | 490         |
|    policy_gradient_loss | -0.00827    |
|    std                  | 0.106       |
|    value_loss           | 0.111       |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | -9.27       |
| time/                   |             |
|    fps                  | 2406        |
|    iterations           | 60          |
|    time_elapsed         | 102         |
|    total_timesteps      | 245760      |
| train/                  |             |
|    approx_kl            | 0.014114834 |
|    clip_fraction        | 0.189       |
|    clip_range           | 0.2         |
|    entropy_loss         | 2.23        |
|    explained_variance   | 0.996       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.0429      |
|    n_updates            | 590         |
|    policy_gradient_loss | 0.000411    |
|    std                  | 0.0793      |
|    value_loss           | 0.0561      |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | -7.47       |
| time/                   |             |
|    fps                  | 2406        |
|    iterations           | 70          |
|    time_elapsed         | 119         |
|    total_timesteps      | 286720      |
| train/                  |             |
|    approx_kl            | 0.013788357 |
|    clip_fraction        | 0.178       |
|    clip_range           | 0.2         |
|    entropy_loss         | 2.68        |
|    explained_variance   | 0.998       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.0557      |
|    n_updates            | 690         |
|    policy_gradient_loss | 0.0023      |
|    std                  | 0.0634      |
|    value_loss           | 0.0415      |
-----------------------------------------


   ✅ Kaydedildi: final_model_Vanilla_seed_42

🎉 PPO (Vanilla) Eğitimi BAŞARIYLA TAMAMLANDI.


In [6]:
paths_framestack = trainer.train_variant("FrameStack")


print("\n🎉 PPOFrameStack Eğitimi BAŞARIYLA TAMAMLANDI.")


🚀 MODEL EĞİTİMİ BAŞLIYOR: FrameStack
   ⚡ Donanım: CPU (MLP model için optimize - Uyarı engellendi)
   🛠️ FrameStack: Aktif (4 frames)
Using cpu device
   🌱 Seed 42 eğitiliyor...
Logging to ../data/logs/Phase-3.1-Comparison_20251125_121126/FrameStack_seed_42_1


Output()

/Users/berkaygurkan/pytorch_m1_env/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/berkaygurkan/pytorch_m1_env/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/berkaygurkan/pytorch_m1_env/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/Users/berkaygurkan/pytorch_m1_env/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | -49.7       |
| time/                   |             |
|    fps                  | 2744        |
|    iterations           | 10          |
|    time_elapsed         | 14          |
|    total_timesteps      | 40960       |
| train/                  |             |
|    approx_kl            | 0.014693832 |
|    clip_fraction        | 0.114       |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.02       |
|    explained_variance   | 0.845       |
|    learning_rate        | 0.0003      |
|    loss                 | 1.88        |
|    n_updates            | 90          |
|    policy_gradient_loss | -0.0185     |
|    std                  | 0.648       |
|    value_loss           | 5.03        |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | -31.4       |
| time/                   |             |
|    fps                  | 2674        |
|    iterations           | 20          |
|    time_elapsed         | 30          |
|    total_timesteps      | 81920       |
| train/                  |             |
|    approx_kl            | 0.015575624 |
|    clip_fraction        | 0.188       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.885      |
|    explained_variance   | 0.827       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.975       |
|    n_updates            | 190         |
|    policy_gradient_loss | -0.0147     |
|    std                  | 0.371       |
|    value_loss           | 1.89        |
-----------------------------------------


...

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | -11.6       |
| time/                   |             |
|    fps                  | 2615        |
|    iterations           | 40          |
|    time_elapsed         | 62          |
|    total_timesteps      | 163840      |
| train/                  |             |
|    approx_kl            | 0.016945172 |
|    clip_fraction        | 0.201       |
|    clip_range           | 0.2         |
|    entropy_loss         | 0.989       |
|    explained_variance   | 0.805       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.181       |
|    n_updates            | 390         |
|    policy_gradient_loss | -0.0153     |
|    std                  | 0.147       |
|    value_loss           | 0.442       |
-----------------------------------------


----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 50         |
|    ep_rew_mean          | -12.5      |
| time/                   |            |
|    fps                  | 2605       |
|    iterations           | 50         |
|    time_elapsed         | 78         |
|    total_timesteps      | 204800     |
| train/                  |            |
|    approx_kl            | 0.01840055 |
|    clip_fraction        | 0.209      |
|    clip_range           | 0.2        |
|    entropy_loss         | 1.29       |
|    explained_variance   | 0.754      |
|    learning_rate        | 0.0003     |
|    loss                 | 0.554      |
|    n_updates            | 490        |
|    policy_gradient_loss | -0.00322   |
|    std                  | 0.126      |
|    value_loss           | 1.08       |
----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | -11.8       |
| time/                   |             |
|    fps                  | 2595        |
|    iterations           | 60          |
|    time_elapsed         | 94          |
|    total_timesteps      | 245760      |
| train/                  |             |
|    approx_kl            | 0.026041728 |
|    clip_fraction        | 0.281       |
|    clip_range           | 0.2         |
|    entropy_loss         | 1.53        |
|    explained_variance   | 0.967       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.16        |
|    n_updates            | 590         |
|    policy_gradient_loss | -0.0104     |
|    std                  | 0.112       |
|    value_loss           | 0.34        |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | -11.3       |
| time/                   |             |
|    fps                  | 2599        |
|    iterations           | 70          |
|    time_elapsed         | 110         |
|    total_timesteps      | 286720      |
| train/                  |             |
|    approx_kl            | 0.019269206 |
|    clip_fraction        | 0.207       |
|    clip_range           | 0.2         |
|    entropy_loss         | 1.84        |
|    explained_variance   | 0.944       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.136       |
|    n_updates            | 690         |
|    policy_gradient_loss | -0.00519    |
|    std                  | 0.0962      |
|    value_loss           | 0.278       |
-----------------------------------------


   ✅ Kaydedildi: final_model_FrameStack_seed_42

🎉 PPOFrameStack Eğitimi BAŞARIYLA TAMAMLANDI.


In [8]:
paths_lstm = trainer.train_variant("LSTM")
print("\n🎉 LSTM augmented obs Eğitimi BAŞARIYLA TAMAMLANDI.")


🚀 MODEL EĞİTİMİ BAŞLIYOR: LSTM
   ⚡ Donanım: mps (Recurrent model için optimize)
Using mps device
   🌱 Seed 42 eğitiliyor...
Logging to ../data/logs/Phase-3.1-Comparison_20251125_121126/LSTM_seed_42_1


Output()

/Users/berkaygurkan/pytorch_m1_env/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/berkaygurkan/pytorch_m1_env/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/berkaygurkan/pytorch_m1_env/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/Users/berkaygurkan/pytorch_m1_env/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | -41.2       |
| time/                   |             |
|    fps                  | 50          |
|    iterations           | 10          |
|    time_elapsed         | 817         |
|    total_timesteps      | 40960       |
| train/                  |             |
|    approx_kl            | 0.020773247 |
|    clip_fraction        | 0.101       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.92       |
|    explained_variance   | -2.1e-05    |
|    learning_rate        | 0.0003      |
|    loss                 | 0.419       |
|    n_updates            | 90          |
|    policy_gradient_loss | -0.0195     |
|    std                  | 0.61        |
|    value_loss           | 3.33        |
-----------------------------------------


KeyboardInterrupt: 

### OOD Test

In [7]:
def evaluate_ood_compare(models_map, n_episodes=20):
    # OOD Ortam: Damping [3.0, 5.0]
    def make_ood_env():
        env = gym.make("Reacher-v5")
        env = Float32ObservationWrapper(env)
        env = RandomDampingWrapper(env, min_damping=3.0, max_damping=5.0) # AĞIR SÜRTÜNME
        env = PreviousActionWrapper(env)
        return env

    env_base = DummyVecEnv([make_ood_env])
    
    results = {}
    
    print(f"\n🧪 OOD TESTİ BAŞLIYOR (Damping: 3.0 - 5.0)")
    
    for name, path in models_map.items():
        print(f"   Testing {name}...")
        
        # FrameStack ise ortamı sarmala, değilse base ortamı kullan
        if name == "FrameStack":
            # FrameStack için DummyVecEnv'i sarmalıyoruz
            eval_env = VecFrameStack(env_base, n_stack=4)
        else:
            eval_env = env_base
            # FrameStack wrapper'ını temizle (varsa)
            if hasattr(eval_env, 'stack_frames'): 
                eval_env = env_base # Basite indirge (VecFrameStack attribute kontrolü)

        # Modeli Yükle
        if name == "LSTM":
            model = RecurrentPPO.load(path, env=eval_env, device="cpu")
        else:
            model = PPO.load(path, env=eval_env, device="cpu")
            
        # Test Döngüsü
        scores = []
        for _ in range(n_episodes):
            obs = eval_env.reset()
            done = False
            total_reward = 0
            lstm_states = None
            episode_starts = np.ones((1,), dtype=bool)
            
            while not done:
                if name == "LSTM":
                    action, lstm_states = model.predict(obs, state=lstm_states, episode_start=episode_starts, deterministic=True)
                else:
                    action, _ = model.predict(obs, deterministic=True)
                
                obs, reward, done, _ = eval_env.step(action)
                total_reward += reward[0]
                episode_starts = done
            scores.append(total_reward)
        results[name] = scores

    # Grafik Çiz
    plt.figure(figsize=(10, 6))
    plt.boxplot(results.values(), labels=results.keys(), patch_artist=True)
    plt.title("OOD Generalization: Unseen Heavy Friction (Damping [3.0, 5.0])")
    plt.ylabel("Total Reward")
    plt.grid(True, alpha=0.3)
    plt.show()

# Modellerin ilk seedlerini alıp test edelim
models_map = {
    "Vanilla": paths_vanilla[0],
    "FrameStack": paths_framestack[0],
    "LSTM": paths_lstm[0]
}
evaluate_ood_compare(models_map)

NameError: name 'paths_lstm' is not defined

### Sudden Change Test

In [ ]:
def test_sudden_change_compare(models_map):
    plt.figure(figsize=(12, 6))
    
    # Tek bir ortam, uzun süreli (100 adım)
    base_gym_env = gym.make("Reacher-v5", max_episode_steps=150)
    base_gym_env = Float32ObservationWrapper(base_gym_env)
    base_gym_env = RandomDampingWrapper(base_gym_env, 0.5, 2.0)
    base_gym_env = PreviousActionWrapper(base_gym_env)
    
    # Asıl fizik motoruna erişim
    real_env_unwrapped = base_gym_env.unwrapped 
    
    for name, path in models_map.items():
        print(f"⚡ Sudden Fault Test: {name}")
        
        # Ortamı hazırla
        if name == "FrameStack":
            # DummyVecEnv + VecFrameStack kombinasyonu
            env = DummyVecEnv([lambda: base_gym_env]) 
            env = VecFrameStack(env, n_stack=4)
        else:
            env = DummyVecEnv([lambda: base_gym_env])

        # Model Yükle
        if name == "LSTM":
            model = RecurrentPPO.load(path, device="cpu")
        else:
            model = PPO.load(path, device="cpu")
            
        # BAŞLANGIÇ DURUMU (Normal Fizik)
        obs = env.reset()
        real_env_unwrapped.set_damping(1.0) 
        
        rewards = []
        lstm_states = None
        episode_starts = np.ones((1,), dtype=bool)
        
        # 100 Adımlık Test
        for t in range(100):
            # --- ANİ ARIZA (t=50) ---
            if t == 50:
                real_env_unwrapped.set_damping(5.0) # AĞIR SÜRTÜNME (Reset Yok)
            
            # Predict
            if name == "LSTM":
                action, lstm_states = model.predict(obs, state=lstm_states, episode_start=episode_starts, deterministic=True)
            else:
                action, _ = model.predict(obs, deterministic=True)
            
            # Step
            obs, reward, done, _ = env.step(action)
            rewards.append(reward[0])
            
            # Reset sinyali gelse bile (time limit), state'i ve ortamı sıfırlama
            # Bu sayede "Adaptasyonu" görüyoruz
            if done:
                episode_starts = np.ones((1,), dtype=bool)
                
        plt.plot(rewards, label=name, linewidth=2.5 if name=="LSTM" else 1.5, alpha=0.8)

    plt.axvline(x=50, color='r', linestyle='--', label="Sudden Fault (x5 Damping)")
    plt.title("Online Adaptation Comparison: Sudden Fault Recovery")
    plt.xlabel("Steps")
    plt.ylabel("Instant Reward")
    plt.legend()
    plt.show()

test_sudden_change_compare(models_map)